In [ ]:
import os
from openai import OpenAI

# model="openai/gpt-oss-20b", 
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key=os.getenv("OPENAI_API_KEY")
)

In [ ]:
response = client.responses.create(
    model="llama3.2:1b-instruct-q8_0",
    input="What to say Hello in Hindi language ?"
)

print(response.output_text)

#### Analyze images and files

[Link](https://developers.openai.com/api/docs/quickstart#analyze-images-and-files)

In [ ]:
#
response = client.responses.create(
    model="llama3.2:1b-instruct-q8_0",
    input=[{
            "role": "user",
            "content": [{
                    "type": "input_text",
                    "text": "India and England are playing cricket match",
                },{
                    "type": "input_text",
                    "text": "Who are playing match ?",
                }]
        }]
)

print(response.output_text)

#### Extend the model with tools



##### Web Search

In [ ]:

response = client.responses.create(
    model="llama3.2:1b-instruct-q8_0",
    tools=[{"type": "web_search"}],
    input="What was a positive news story from today?"
)

print(response.output_text)

##### File Search

In [ ]:
#
response = client.responses.create(
    model="llama3.2:1b-instruct-q8_0",
    input="What is deep research by OpenAI?",
    tools=[{
        "type": "file_search",
        "vector_store_ids": ["<vector_store_id>"]
    }]
)
print(response)

##### Functional Call

In [ ]:
tools = [{
        "type": "function",
        "name": "get_weather",
        "description": "Get current temperature for a given location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City and country e.g. Bogotá, Colombia",
                }
            },
            "required": ["location"],
            "additionalProperties": False,
        },
        "strict": True,
    }]

response = client.responses.create(
    model="llama3.2:1b-instruct-q8_0",
    input=[
        {"role": "user", "content": "What is the weather like in Paris today?"},
    ],
    tools=tools,
)

print(response.output[0].to_json())

##### Remote MCP Server


In [ ]:
#
resp = client.responses.create(
    model="llama3.2:1b-instruct-q8_0",
    tools=[
        {
            "type": "mcp",
            "server_label": "dmcp",
            "server_description": "A Dungeons and Dragons MCP server to assist with dice rolling.",
            "server_url": "https://dmcp-server.deno.dev/sse",
            "require_approval": "never",
        },
    ],
    input="Roll 2d4+1",
)

print(resp.output_text)

##### Stream responses and build realtime apps



In [ ]:
#
stream = client.responses.create(
    model="llama3.2:1b-instruct-q8_0",
    input=[
        {
            "role": "user",
            "content": "Say 'double bubble bath' ten times fast.",
        },
    ],
    stream=True,
)

for event in stream:
    print(event)

##### from openai import OpenAI


In [ ]:
# pip install -U openai-agents

from agents import Agent, Runner, Model, RunConfig
import asyncio


spanish_agent = Agent(
    name="Spanish agent",
    instructions="You only speak Spanish.",
    model="llama3.2:1b-instruct-q8_0",
)

english_agent = Agent(
    name="English agent",
    instructions="You only speak English",
    model="llama3.2:1b-instruct-q8_0",
)

triage_agent = Agent(
    name="Triage agent",
    instructions="Handoff to the appropriate agent based on the language of the request.",
    handoffs=[spanish_agent, english_agent],
    model="llama3.2:1b-instruct-q8_0",
)


async def main():
    result = await Runner.run(triage_agent, input="Hola, ¿cómo estás?")
    print(result.final_output)


import nest_asyncio
nest_asyncio.apply()
asyncio.run(main()) # Now will work

#if __name__ == "__main__":
#    asyncio.run(main())

Error getting response: Error code: 400 - {'error': {'message': "The requested model 'llama3.2:1b-instruct-q8_0' does not exist.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_found'}}. (request_id: req_2d6c107da0fb429dbeab2c333431152c)


BadRequestError: Error code: 400 - {'error': {'message': "The requested model 'llama3.2:1b-instruct-q8_0' does not exist.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_found'}}

In [ ]:
from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model="ollama:llama3.2:1b-instruct-q8_0",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# Run the agent
response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)

for m in response['messages'] :
    m.pretty_print()
